In [ ]:
# Si importa la classe ReachySDK dal modulo reachy_skd
from reachy_sdk import ReachySDK 

# Si importa la libreria numpy
import numpy as np

# Si importa la libreria time
import time

# Si importa la funzione goto dal modulo reachy_sdk
from reachy_sdk.trajectory import goto

# Si importa la classe InterpolationMode dal modulo reachy_sdk
from reachy_sdk.trajectory.interpolation import InterpolationMode

# Si importa il modulo Image dal pacchetto PIL
from PIL import Image

# Si importa la classe OwlVitWrapper dal modulo object_detection all'interno del pacchetto vision_models della libreria pollen_vision
from pollen_vision.vision_models.object_detection import OwlVitWrapper

# Si importa le classe get_bboxes dal modulo utils all'interno del pacchetto vision_models della libreria pollen_vision
from pollen_vision.vision_models.utils import get_bboxes

# Si importa le classi Annotator dal modulo utils all'interno del pacchetto vision_models della libreria pollen_vision
from pollen_vision.vision_models.utils import Annotator

# Si importa la classe MobileSamWrapper dal modulo object_segmentation all'interno del pacchetto vision_models della libreria pollen_vision
from pollen_vision.vision_models.object_segmentation import MobileSamWrapper


In [2]:
# Si stabilisce una connessione con Reachy 
reachy = ReachySDK(host='localhost')

In [ ]:
# Si controllano se tutte le articolazioni di Reachy sono rilevate
reachy.joints

In [ ]:
# Si controlla se tutti i sensori di forza di Reachy sono rilevati
reachy.force_sensors

In [5]:
# Si fa abbassare la testa di Reachy per fare in modo che rivolga lo sguardo davanti la tazza riposta nel tavolo
look_down = reachy.head.look_at(
    x=0.5,
    y=0,
    z=-0.4,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

In [ ]:
# Si assegna alla variabile "img" l'ultimo frame dall'occhio destro di Reachy
img = Image.fromarray(reachy.right_camera.last_frame[:,:,::-1])
img

In [ ]:
# Si crea un'istanza della classe OwlVitWrapper e si assegna alla variabile "object_detection_wrapper"
object_detection_wrapper = OwlVitWrapper()

In [ ]:
'''
Si utilizza il modello object_detection_wrapper per eseguire la rilevazione degli oggetti sull'immagine im, 
cercando "cups" utilizzando la zero-shot object detection. 
I risultati della rilevazione vengono assegnati alla variabile "predictions".
'''
predictions = object_detection_wrapper.infer(
    im=np.array(img), candidate_labels=["mug"], detection_threshold=0.06
)
predictions

In [9]:
'''
Si utilizza il modello get_bboxes per ottenere le caselle delimitatrici (bounding boxes)
dai risultati della rilevazione degli ogetti contenuti nella variabile "predictions".
Le bounding boxes vengono assegnate alla variabile "bboxes".
'''
bboxes = get_bboxes(predictions)

In [10]:
# Si crea un'istanza della classe Annotator e si assegna alla variabile "annotator"
annotator = Annotator()

In [ ]:
'''
Si utilizza l'oggetto "annotator" per aggiungere annotazioni all'immagine "im",
utilizzando i risultati della rilevazione degli oggetti contenuti nella variabile "prediction".
L'immagine risultante viene assegnata alla variabile "img_annotated"
'''
img_annotated = annotator.annotate(im=img, detection_predictions=predictions)
Image.fromarray(img_annotated)

In [12]:
# Si crea un'istanza della classe MobileSamWrapper e si assegna alla variabile "object_segmentation_wrapper"
object_segmentation_wrapper = MobileSamWrapper()

In [13]:
'''
Si utilizza il modello sam per eseguire la segmentazione degli oggetti sull'immagine im,
utilizzando le bounding boxes ottenute dalla rilevazione degli oggetti
utilizzando la zero-shot object segmentation. 
I risultati della segmentazione vengono assegnati alla variabile "masks".
'''
masks = object_segmentation_wrapper.infer(im=img, bboxes=bboxes)

In [ ]:
'''
Si utilizza l'oggetto "annotator" per aggiungere annotazioni all'immagine "im",
utilizzando i risultati della rilevazione degli oggetti contenuti nella variabile "prediction"
e le maschere di segmentazione contenute nella variabile "masks".
L'immagine risultante viene assegnata alla variabile "img_annotated"
'''
img_annotated = annotator.annotate(im=img, detection_predictions=predictions, masks=masks)
Image.fromarray(img_annotated)